In [59]:
# [CELL 1] STUDENT CONFIGURATION
from pathlib import Path
import numpy as np
import pandas as pd
LAST_NAME = "Celo"
STUDENT_ID = "TUPM-26-1941"
id_digits = "".join(d for d in STUDENT_ID if d.isdigit())
assert id_digits, "Student ID must contain digits."
SEED_DIGIT = int(id_digits[-1])
ID_SUM = sum(map(int, id_digits))
NAME_LENGTH = len(LAST_NAME)
RANDOM_SEED = int(id_digits) % (2**32)
verification_value = ID_SUM * NAME_LENGTH + SEED_DIGIT
np.set_printoptions(precision=4, suppress=True)

print("Student:", LAST_NAME, STUDENT_ID)
print("Seed Digit:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Name Length:", NAME_LENGTH)
print("Random Seed:", RANDOM_SEED)
print("Verification Value:", verification_value)


Student: Celo TUPM-26-1941
Seed Digit: 1
ID Sum: 23
Name Length: 4
Random Seed: 261941
Verification Value: 93


In [60]:
# [CELL 2] VERIFY WORKSPACE
WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"

DATA_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

measurement_path = DATA_DIR / "secom.data"
label_path = DATA_DIR / "secom_labels.data"
description_path = DATA_DIR / "secom.names"

for path in [measurement_path, label_path, description_path]:
    assert path.is_file(), f"Missing file: {path}"

print("Workspace:", WORK_DIR.resolve())
print("All dataset files verified.")

Workspace: C:\Users\admin\Lab7_NumPy_Pandas_Celo
All dataset files verified.


In [61]:
# Load the SECOM measurement matrix and identify features containing missing or constant observed values.
raw_measurements = np.genfromtxt(measurement_path, dtype=float)
assert raw_measurements.ndim == 2

missing = np.isnan(raw_measurements)
observed = (~missing).sum(axis=0)

all_missing = observed == 0

observed_min = np.where(
missing, np.inf, raw_measurements
).min(axis=0)

observed_max = np.where(
missing, -np.inf, raw_measurements
).max(axis=0)

constant = (observed > 0) & (observed_min == observed_max)
usable_indices = np.flatnonzero(~(all_missing | constant))

print("Features Containing Missing Values:", np.isnan(raw_measurements).sum())
print("Features Containing Constant Values:", constant.sum())


Features Containing Missing Values: 41951
Features Containing Constant Values: 116


In [62]:
# Select 5 + (SEED_DIGIT % 3) usable features reproducibly.
rng = np.random.default_rng(RANDOM_SEED)
selected_indices = rng.choice(usable_indices, size=min(5 + (SEED_DIGIT % 3), len(usable_indices)), replace=False)
indices = raw_measurements[:, selected_indices].copy()

In [63]:
# Calculate feature-wise means, medians, sample standard deviations, and missing percentages using NumPy.
feature_medians = np.nanmedian(indices, axis=0)
feature_means = np.nanmean(indices, axis=0)
feature_sds = np.nanstd(indices, axis=0, ddof=1)
feature_missing_percentages = (np.isnan(indices).sum(axis=0) / indices.shape[0]) * 100

selected_features = [f"Feature_{i:03d}" for i in selected_indices]

print("Selected Features:", selected_features)
print("Feature-Wise Means:", np.round(feature_means, 4))
print("Feature Medians:", np.round(feature_medians, 4))
print("Sample Standard Deviations:", np.round(feature_sds, 4))
print("Missing Percentages (%):", np.round(feature_missing_percentages, 2))


Selected Features: ['Feature_572', 'Feature_223', 'Feature_570', 'Feature_211', 'Feature_143', 'Feature_383']
Feature-Wise Means: [ 28.4502 122.8466 530.5236   0.0568   0.0042   0.541 ]
Feature Medians: [  8.65   119.436  532.3982   0.0532   0.0039   0.3726]
Sample Standard Deviations: [86.3047 55.156  17.4997  0.025   0.0013  1.341 ]
Missing Percentages (%): [ 0.    0.    0.    1.53  0.57 64.96]


In [64]:
# Compute robust standardized values using the median and median absolute deviation (MAD). Handle features with zero MAD safely.
feature_medians = np.nanmedian(indices, axis=0)
feature_mads = np.nanmedian(np.abs(indices - feature_medians), axis=0)
mads = np.where(feature_mads == 0, 1, feature_mads)
robust_standardized = np.where(feature_mads > 0, (indices - feature_medians) / mads, 0)  


In [65]:
# Identify measurements with absolute robust standardized values greater than 3.5.
flagged_indices = np.abs(robust_standardized) > 3.5
count_flagged = flagged_indices.sum(axis=0)
flagged_percentages = (count_flagged / (~np.isnan(indices)).sum(axis=0)) * 100

# Rank features by their percentage of flagged observations and generate a CSV summary.
summary_df = pd.DataFrame({
    "Feature": selected_features,
    "Mean": feature_means,
    "Median": feature_medians,
    "Sample_SD": feature_sds,
    "Missing_Percentage": feature_missing_percentages,
    "MAD": feature_mads,
    "Flagged_Count": count_flagged,
    "Flagged_Percentage": flagged_percentages
}).sort_values("Flagged_Percentage", ascending=False).reset_index(drop=True)
 
summary_df.to_csv('measurement_quality.csv', index=False)  

assert np.isclose(count_flagged.sum(), np.count_nonzero(flagged_indices))

In [66]:
# [Assessment Data] Output Parameter
print("Selected Features:", ", ".join(selected_features))
print(f"Highest Missing Percentage: {summary_df['Missing_Percentage'].max():.2f}%")
print("Feature with Highest Flagged Percentage:", summary_df.loc[0, "Feature"])
print("Total Flagged Observations:", int(count_flagged.sum()))
print("Vectorized Results Verified:", True)

Selected Features: Feature_572, Feature_223, Feature_570, Feature_211, Feature_143, Feature_383
Highest Missing Percentage: 64.96%
Feature with Highest Flagged Percentage: Feature_383
Total Flagged Observations: 535
Vectorized Results Verified: True
